# Overview of anomalies

In [1]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from anomaly.constants import GALAXY_LINES
from anomaly.utils import specobjid_to_idx
from anomaly.utils import VelocityFilter
from anomaly.utils import AnomalyOverlapAnalyzer
from autoencoders.ae import AutoEncoder

from sdss.metadata import MetaData

meta = MetaData()

# Constants

In [3]:
se_cols = [
    'mse',
    'mse_filter_250',
    # 'mse_filter_300',
    'mse_97',
    # 'mse_95',
    'mse_filter_250_97',
    # 'mse_filter_250_95',
    # 'mse_filter_300_97', 'mse_filter_300_95'
]

se_rank_cols = [
    f"rank_{col}" for col in se_cols
]

# ----------------------------------------------
rse_cols = [
    f"{col}_rel" for col in se_cols
]

rse_rank_cols = [
    f"rank_{col}_rel" for col in se_cols
]
# ----------------------------------------------
se_family = [
    'mse',
    'mse_97',
    # 'mse_95',
    'mse_filter_250',
    # 'mse_filter_300',
    'mse_filter_250_97',
    # 'mse_filter_250_95',
    # 'mse_filter_300_97', 'mse_filter_300_95'
]

rse_family = [
    f'{col}_rel' for col in se_family
]

# Custom functions

## IDs top anomalies

In [4]:
def get_ids(score, df, quantile=99, n_top=None, use_ntop=False):

    if use_ntop is False:
        
        quantile *= 0.01
        thresh = df[score].quantile(quantile)
        ids = set(df[df[score] > thresh].index)
        
    else:

        ids = set(
            df[score].sort_values(
                ascending=False
            ).iloc[:n_top].index
        )

    return ids

In [5]:
def top_unique_ids(scores_df, scores_list, quantile=99, n_top=None, use_ntop=False):

    ids_top_dict = {}

    for score in scores_list:

        ids_top_dict[score] = get_ids(
            score=score,
            df=scores_df,
            quantile=quantile,
            n_top=n_top,
            use_ntop=use_ntop
        )

    n_top = len(ids_top_dict[score])

    unique_ids_dict = AnomalyOverlapAnalyzer.get_unique_ids(
        ids_dict=ids_top_dict, score_list=scores_list
    )

    for score in scores_list:
        n_unique = len(unique_ids_dict[score])

        unique_pct = n_unique/n_top*100
        
        print(f"Unique to {score}:\n{n_unique} --> {unique_pct:.4f}%")

    return unique_ids_dict, ids_top_dict


In [6]:
def top_common_ids(scores_df, scores_list, quantile=99, n_top=None, use_ntop=False):

    ids_top_dict = {}

    for score in scores_list:

        ids_top_dict[score] = get_ids(
            score=score,
            df=scores_df,
            quantile=quantile,
            n_top=n_top,
            use_ntop=use_ntop
        )

    n_top = len(ids_top_dict[score])

    common_ids_dict = AnomalyOverlapAnalyzer.get_core_common_ids(
        ids_dict=ids_top_dict, score_list=scores_list
    )

    n_common = len(common_ids_dict)
    common_pct = n_common/n_top*100 
    print(f"N common:\n{n_common} -- > {common_pct:4f}%")

    return common_ids_dict, ids_top_dict

## Figures

In [7]:
def anomaly_plot(wave, specs, objids, ranks, save_to):

    fig, ax = plt.subplots(
        figsize=(10, 5)
    )

    for spec, objid, rank in zip(specs, objids, ranks):

        print(f'Rank {rank:03d}', end='\r')

        ax.clear()

        ax.plot(wave, spec, color="black", label=f'Rank: {rank}')

        ax.minorticks_on()
        ax.set_xlabel(r"$\lambda$ [nm]")
        ax.set_title(f"Object ID: {objid}")

        ax.legend(
            loc='upper left',
            frameon=False,
        )

        fig.savefig(
            f"{save_to}/{rank:03d}_{objid}.jpeg",
            bbox_inches='tight'
        )

    plt.close(fig)

# Config

## Directories

In [8]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
scores_dir = f"{data_dir}/scores"
models_dir = f"{data_dir}/models"
bin_id = 'bin_01'
#
ch_4_dir = f"{thesis_dir}/chapters/04_figures"
os.makedirs(f"{ch_4_dir}/{bin_id}", exist_ok=True)

## Data

In [9]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)

In [10]:
score_df = pd.read_csv(
    f"{scores_dir}/{bin_id}/scores_{bin_id}.csv.gz",
    index_col='specobjid'
)

rank = np.arange(score_df.shape[0]) + 1
score_rank_df = score_df.copy()
# score_rank_df
for col in score_df.columns:

    index_sorted = score_df.sort_values(
        by=col, ascending=False
    ).index

    score_rank_df.loc[index_sorted, f'rank_{col}'] = rank
    score_rank_df[f'rank_{col}'].astype(int)

n_spec = score_df.shape[0]
n_top_1_pct = int(n_spec*0.01)
n_top_1_pct, n_spec

(1818, 181850)

In [11]:
score = 'mse_97'
score_rank_df[[score, f'rank_{score}']].sort_values(
    by=score, ascending=False
).head(10)

,mse_97,rank_mse_97
specobjid,,
2034543258029287424,17.920967,1.0
1137241688603912192,10.646924,2.0
650786678071388160,10.600596,3.0
1034778745740748800,9.506108,4.0
572079513039562752,9.356786,5.0
2581793002573817856,9.179527,6.0
1623712368703858688,8.978367,7.0
1531300890005235712,8.945759,8.0
2402769422277175296,8.889236,9.0


# Figs top anomalies

In [12]:
# ```python
n_top = 1000
all_scores = se_cols + rse_cols
plt.ioff()

for score in all_scores:

    specids_top_1 = score_df[score].sort_values(
        ascending=False
    ).index.to_numpy()[:n_top]

    ranks = np.zeros(n_top).astype(int)

    specs_top_1 = np.empty((n_top, wave.size))

    for i, objid in enumerate(specids_top_1):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs_top_1[i, :] = spectra[spec_idx, :]

        ranks[i] = i

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs_top_1,
        objids=specids_top_1, ranks=ranks,
        save_to=save_to
    )
# ```

# No free lunch theorem

## IDs per score

In [13]:
ids_top_dict = {}

all_scores = se_cols + rse_cols

for score in all_scores:

    ids_top_dict[score] = get_ids(
        score=score,
        df=score_df.copy(),
        quantile=99,
        n_top=1000,
        use_ntop=False
    )

n_top_1 = len(ids_top_dict[score])
n_top_1

1819

# Distinct IDs

## SE family

In [14]:
se_unique_ids_dict, se_ids_top_dict = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=se_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

Unique to mse:
443 --> 24.3540%
Unique to mse_filter_250:
307 --> 16.8774%
Unique to mse_97:
66 --> 3.6284%
Unique to mse_filter_250_97:
156 --> 8.5761%


In [15]:
score = 'mse'
unique_score_ids = list(se_unique_ids_dict[score])
score_rank_df.loc[
    unique_score_ids, [score, f'rank_{score}']
].sort_values(by=score, ascending=False).head()

,mse,rank_mse
specobjid,,
2917311181691578368,28.962301,81.0
1974864502398674944,28.317249,83.0
1533515856464603136,27.436001,89.0
2389268249188526080,19.650555,161.0
1630310814034454528,19.564099,166.0


### Figs unique per SE

In [16]:
plt.ioff()

for score in se_cols:

    specids = list(se_unique_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/unique_se/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

## RSE family

In [17]:
rse_unique_ids_dict, chi_ids_top_dict = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=rse_cols,
    quantile=99,
    # n_top=1000, use_ntop=True
)

Unique to mse_rel:
259 --> 14.2386%
Unique to mse_filter_250_rel:
150 --> 8.2463%
Unique to mse_97_rel:
114 --> 6.2672%
Unique to mse_filter_250_97_rel:
144 --> 7.9164%


In [18]:
score = 'mse_rel'
unique_res_ids = list(rse_unique_ids_dict[score])
score_rank_df.loc[
    unique_res_ids, [score, f'rank_{score}']
].sort_values(by=score, ascending=False).head()

,mse_rel,rank_mse_rel
specobjid,,
746583274654033920,29.425823,8.0
1533515856464603136,20.984189,19.0
1993005341155026944,19.766287,21.0
2917311181691578368,17.508401,23.0
2389268249188526080,17.226661,24.0


### Figs unique per RES

In [19]:
plt.ioff()

for score in rse_cols:

    specids = list(rse_unique_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/unique_rse/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

## All scores

In [20]:
all_scores = se_cols + rse_cols

all_unique_ids_dict, all_ids_top_dict = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=all_scores,
    quantile=99,
    # n_top=1000, use_ntop=True
)

Unique to mse:
283 --> 15.5580%
Unique to mse_filter_250:
206 --> 11.3249%
Unique to mse_97:
50 --> 2.7488%
Unique to mse_filter_250_97:
75 --> 4.1231%
Unique to mse_rel:
30 --> 1.6493%
Unique to mse_filter_250_rel:
99 --> 5.4426%
Unique to mse_97_rel:
48 --> 2.6388%
Unique to mse_filter_250_97_rel:
98 --> 5.3876%


In [21]:
score = 'mse'
unique_all_ids = list(all_unique_ids_dict[score])
score_rank_df.loc[
    unique_all_ids, [score, f'rank_{score}']
].sort_values(by=score, ascending=False).head()

,mse,rank_mse
specobjid,,
2351039867665803264,18.321872,194.0
2814767154509932544,17.651639,209.0
2439882887735044096,17.563054,211.0
333380754920728576,15.324141,267.0
1111371552277948416,15.281632,269.0


### Figs unique among all

In [22]:
all_scores = se_cols + rse_cols
plt.ioff()

for score in all_scores:

    specids = list(all_unique_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/unique_all/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

# Common IDs

## SE family

In [23]:
se_common_ids, se_ids_top_dict = top_common_ids(
    scores_df=score_df.copy(), scores_list=se_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N common:
569 -- > 31.280924%


In [24]:
scores = ['mse', 'mse_rel', 'mse_97', 'mse_97_rel']
rank_scores = [f'rank_{s}' for s in scores]
common_se_ids = list(se_common_ids)
score_rank_df.loc[
    common_se_ids, scores + rank_scores
].sort_values(by='mse', ascending=False).head()

,mse,mse_rel,mse_97,mse_97_rel,rank_mse,rank_mse_rel,rank_mse_97,rank_mse_97_rel
specobjid,,,,,,,,
1826320481486137344,115.083783,26.872182,6.495171,6.176242,2.0,14.0,154.0,298.0
378393936579815424,104.726463,15.857475,5.349991,6.190218,3.0,33.0,1754.0,294.0
2571705257897781248,86.060266,70.844025,7.353142,6.873755,5.0,1.0,46.0,94.0
1518887124890314752,85.133051,65.297358,6.298750,5.807904,6.0,2.0,217.0,661.0
2169610825470339072,82.808935,16.247141,7.355646,7.106146,7.0,29.0,45.0,71.0


### Figs common among SE

In [25]:
specids = list(se_common_ids)
specids = np.array(specids, dtype=int)

ranks = score_rank_df.loc[
    specids, f'rank_mse'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/{bin_id}/figs/common_ses"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)

## RSE Family

In [26]:
rse_common_ids, res_ids_top_dict = top_common_ids(
    scores_df=score_df.copy(), scores_list=rse_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N common:
900 -- > 49.477735%


In [27]:
scores = ['mse', 'mse_rel', 'mse_97', 'mse_97_rel']
rank_scores = [f'rank_{s}' for s in scores]

common_rse_ids = list(rse_common_ids)

score_rank_df.loc[
    common_rse_ids, scores + rank_scores
].sort_values(by='mse_rel', ascending=False).head()

,mse,mse_rel,mse_97,mse_97_rel,rank_mse,rank_mse_rel,rank_mse_97,rank_mse_97_rel
specobjid,,,,,,,,
2571705257897781248,86.060266,70.844025,7.353142,6.873755,5.0,1.0,46.0,94.0
1518887124890314752,85.133051,65.297358,6.298750,5.807904,6.0,2.0,217.0,661.0
640804485622425600,65.719043,54.310463,5.800605,5.822665,18.0,3.0,580.0,637.0
2802398742823593984,56.309645,41.651420,7.535682,6.621769,24.0,4.0,35.0,141.0
2034543258029287424,54.535352,41.547469,17.920967,17.774876,26.0,5.0,1.0,1.0


### Figs common among RSE

In [28]:
specids = list(rse_common_ids)
specids = np.array(specids, dtype=int)

ranks = score_rank_df.loc[
    specids, f'rank_mse_rel'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/{bin_id}/figs/common_rses/"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)

## All scores

In [29]:
all_scores = se_cols + rse_cols 
all_common_ids, all_ids_top_dict = top_common_ids(
    scores_df=score_df.copy(), scores_list=all_scores,
    quantile=99,
    n_top=None, use_ntop=False
)

N common:
383 -- > 21.055525%


In [30]:
scores = ['mse', 'mse_rel', 'mse_97', 'mse_97_rel']
rank_scores = [f'rank_{s}' for s in scores]

common_all_ids = list(all_common_ids)
score_rank_df.loc[
    common_all_ids, scores + rank_scores
].sort_values(by='mse', ascending=False).head(10)

,mse,mse_rel,mse_97,mse_97_rel,rank_mse,rank_mse_rel,rank_mse_97,rank_mse_97_rel
specobjid,,,,,,,,
1826320481486137344,115.083783,26.872182,6.495171,6.176242,2.0,14.0,154.0,298.0
378393936579815424,104.726463,15.857475,5.349991,6.190218,3.0,33.0,1754.0,294.0
2571705257897781248,86.060266,70.844025,7.353142,6.873755,5.0,1.0,46.0,94.0
1518887124890314752,85.133051,65.297358,6.298750,5.807904,6.0,2.0,217.0,661.0
2169610825470339072,82.808935,16.247141,7.355646,7.106146,7.0,29.0,45.0,71.0
2897076043246495744,79.455599,15.856667,8.780182,7.800237,9.0,34.0,10.0,36.0
1413058308851394560,72.328764,14.022923,7.440662,8.315316,11.0,52.0,41.0,18.0
640804485622425600,65.719043,54.310463,5.800605,5.822665,18.0,3.0,580.0,637.0
1623712368703858688,62.251807,12.974128,8.978367,8.434079,20.0,73.0,7.0,15.0


### Figures

In [31]:
all_scores = se_cols + rse_cols
plt.ioff()


specids = list(all_common_ids)
specids = np.array(specids, dtype=int)

ranks = score_rank_df.loc[
    specids, 'rank_mse'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/{bin_id}/figs/common_all"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)